# Lazarus Group 블록체인 해킹 — 통합 실습 노트북

> Bybit 해킹 (2025-02, 약 14.6억 달러, 역대 최대) 의 *공개 데이터*를 처음부터 끝까지 따라가는 한 권의 노트북.

## 학습 흐름

| 단계 | 내용 | 어느 Lab |
|------|------|---------|
| Step 1 | 환경 준비 + API 키 | — |
| Step 2 | Bybit Exploiter 의 1-hop 자금 추적 (ETH + ERC-20) | Lab 1 |
| Step 3 | ERC-20 노이즈에서 Address Poisoning 폭로 | Lab 5 응용 |
| Step 4 | OFAC 제재 주소와 1-hop 대조 | Lab 2 |
| Step 5 | 행동 패턴 휴리스틱으로 위험도 점수화 | Lab 3 |
| Step 6 | 2-hop 추적 — 한 단계 더 깊이 | Lab 1 확장 |
| Step 7 | 2-hop 결과를 OFAC 와 다시 대조 | Lab 2 확장 |
| Step 8 | NetworkX 그래프 — 실데이터 peeling chain | Lab 6 |

## 윤리 고지 (강의 시작 시 5분 강조)

- 본 노트북은 **공개 블록체인 데이터**와 **공개 정부/커뮤니티 자료** 만 사용한다.
- 어떤 코드도 자산 이전·키 사용·익스플로잇 생성·피싱 사이트 구축을 수행하지 **않는다**.
- 본 자료를 사용해 타인의 시스템·자산을 침해하면 형법 제347조의2(컴퓨터등 사용 사기), 정보통신망법 제48조 위반이다.
- Lazarus 의 기법을 *공부* 하는 이유는 *방어* 하기 위해서이지, 따라하기 위해서가 아니다.

---


## Step 1 — 환경 준비

Colab 에 거의 모든 패키지가 기본 설치되어 있다. 필요한 것만 추가 설치한다.


In [ ]:
!pip -q install requests pandas networkx matplotlib


### Etherscan API 키 입력

[Etherscan](https://etherscan.io/myapikey) 에서 무료 키 발급 후 아래 셀에서 입력.

> 2025-08-15 부로 Etherscan **V1 API 는 폐기**, V2 (`/v2/api`) + `chainid` 파라미터가 필수.
> V2 부터는 placeholder 키로는 동작하지 않으니 반드시 발급받은 키를 입력해야 한다.


In [ ]:
import getpass, os
key = getpass.getpass("Etherscan API key (필수): ").strip()
os.environ["ETHERSCAN_API_KEY"] = key or "YourApiKeyToken"
print("[+] API key set:", "(custom key)" if key else "⚠️ NO KEY — V2는 키 없이는 동작하지 않습니다")


## Step 2 — Bybit Exploiter 의 1-hop 자금 추적 (Lab 1)

**시드 주소**: `0x47666fab8bd0ac7003bce3f5c3585383f09486e2` — FBI PSA (2025-02-26) 가 공개한 1차 수령 주소.

이 주소에서 *나간* ETH 와 ERC-20 토큰을 모두 가져와 수신자별로 합산한다.


In [ ]:
from __future__ import annotations
import os, time
from dataclasses import dataclass
from typing import Iterable
import requests
import pandas as pd

# ----- Etherscan V2 -----
ETHERSCAN_V2_BASE = "https://api.etherscan.io/v2/api"
ETH_MAINNET_CHAINID = 1
API_KEY = os.environ.get("ETHERSCAN_API_KEY", "YourApiKeyToken")

SEED_ADDRESS = "0x47666fab8bd0ac7003bce3f5c3585383f09486e2"
SEED_LABEL = "Bybit Exploiter (1차 수령)"


@dataclass
class Transfer:
    block: int
    timestamp: int
    tx_hash: str
    from_addr: str
    to_addr: str
    amount: float
    asset: str
    is_outbound: bool


def _get(action: str, address: str, max_count: int = 1000) -> list[dict]:
    params = {
        "chainid": ETH_MAINNET_CHAINID,
        "module": "account",
        "action": action,
        "address": address,
        "startblock": 0, "endblock": 99999999,
        "page": 1, "offset": max_count,
        "sort": "desc",
        "apikey": API_KEY,
    }
    r = requests.get(ETHERSCAN_V2_BASE, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()
    if data.get("status") != "1":
        msg = data.get("message", "?")
        result = data.get("result", "")
        if msg == "No transactions found" or result == []:
            return []
        # 학생이 원인을 즉시 보게
        print(f"  ⚠️ Etherscan: status={data.get('status')}, message={msg}")
        if isinstance(result, str) and result:
            print(f"     result: {result[:200]}")
        return []
    return data["result"]


def get_normal_txs(address, max_count=1000): return _get("txlist", address, max_count)
def get_erc20_txs (address, max_count=1000): return _get("tokentx", address, max_count)


def to_eth_transfers(address, raw):
    out, al = [], address.lower()
    for tx in raw:
        wei = int(tx.get("value", "0"))
        out.append(Transfer(int(tx["blockNumber"]), int(tx["timeStamp"]),
                            tx["hash"], tx["from"], tx["to"] or "",
                            wei/10**18, "ETH", tx["from"].lower()==al))
    return out


def to_erc20_transfers(address, raw):
    out, al = [], address.lower()
    for tx in raw:
        try: dec = int(tx.get("tokenDecimal","18"))
        except ValueError: dec = 18
        amt = int(tx.get("value", "0"))
        out.append(Transfer(int(tx["blockNumber"]), int(tx["timeStamp"]),
                            tx["hash"], tx["from"], tx["to"] or "",
                            amt/(10**dec), tx.get("tokenSymbol","ERC20"),
                            tx["from"].lower()==al))
    return out


def trace_outbound(seed: str, top_n: int = 50) -> tuple[pd.DataFrame, pd.DataFrame]:
    eth_raw   = get_normal_txs(seed, max_count=10_000)
    erc20_raw = get_erc20_txs(seed,  max_count=10_000)

    eth_t   = [t for t in to_eth_transfers(seed, eth_raw)     if t.is_outbound and t.amount > 0]
    erc20_t = [t for t in to_erc20_transfers(seed, erc20_raw) if t.is_outbound and t.amount > 0]

    def aggregate(transfers):
        if not transfers:
            return pd.DataFrame(columns=["receiver","asset","tx_count","total","first_ts","last_ts"])
        df = pd.DataFrame([t.__dict__ for t in transfers])
        agg = (df.groupby(["to_addr","asset"])
                 .agg(tx_count=("tx_hash","count"),
                      total=("amount","sum"),
                      first_ts=("timestamp","min"),
                      last_ts=("timestamp","max"))
                 .reset_index()
                 .rename(columns={"to_addr":"receiver"})
                 .sort_values("total", ascending=False)
                 .head(top_n))
        agg["first_ts"] = pd.to_datetime(agg["first_ts"], unit="s", utc=True)
        agg["last_ts"]  = pd.to_datetime(agg["last_ts"],  unit="s", utc=True)
        return agg

    return aggregate(eth_t), aggregate(erc20_t)


### 실행

In [ ]:
print(f"[seed] {SEED_LABEL}\n        {SEED_ADDRESS}\n")
eth_df, erc20_df = trace_outbound(SEED_ADDRESS, top_n=50)
print(f"  ETH 출금 1-hop 수신자: {len(eth_df)} 개")
print(f"  ERC-20 출금 1-hop 수신자: {len(erc20_df)} 개")

print("\n── ETH 출금 ──")
display(eth_df)

print("\n── ERC-20 출금 (raw, 노이즈 포함) ──")
display(erc20_df)


### 관찰

ETH 출금을 보면 **모든 금액이 정확히 10,000 ETH**, **2025-02-21 14:56 ~ 15:52 의 약 1시간**, 특히 **15:48–15:52 의 5분에 집중**되어 있다.
401,000 ETH 총 탈취액이 약 40개 EOA × 10,000 ETH 로 균등 분배된 *자동화된* 1차 분산이다.

ERC-20 표는 좀 이상하다 — 토큰 심볼이 `stETH`, `stEТH`, `ѕтETН`, `ЕТН` 등으로 *비슷하지만 다르다*. 다음 단계에서 폭로한다.


## Step 3 — ERC-20 노이즈의 정체: Address Poisoning + Token Spoofing

위 ERC-20 표를 자세히 보면 `stETH` (라틴) 와 `stEТH` (키릴 `Т`) 가 섞여 있다. 누구나 임의의 ERC-20 컨트랙트를 만들어서 `Transfer(from=Bybit_Exploiter, to=lookalike_address, amount=...)` 이벤트를 송출할 수 있다 — **실제로 Bybit Exploiter 가 보낸 적은 없는 토큰이다**.

이건 사기꾼들이 Bybit Exploiter 주소가 유명해진 걸 이용해 *그 주소를 둘러싼 사람들*을 노린 Address Poisoning 시도다. Lab 5 의 homograph 탐지 로직을 그대로 가져와 진짜 / 가짜를 분리한다.


In [ ]:
# 확장된 LOOKALIKE_MAP — 키릴 + 그리스 + 풀너비 등 라틴 시각유사 문자
LOOKALIKE_MAP = {
    # 키릴 (소문자)
    "а":"a","в":"v","е":"e","ѕ":"s","і":"i","ј":"j","о":"o","р":"p",
    "с":"c","т":"t","у":"y","х":"x","ѵ":"v","ԁ":"d","ɡ":"g",
    # 키릴 (대문자)
    "А":"A","В":"B","Е":"E","З":"3","К":"K","М":"M","Н":"H","О":"O",
    "Р":"P","С":"C","Т":"T","У":"Y","Х":"X","Ѕ":"S","І":"I","Ј":"J",
    # 그리스
    "ο":"o","ν":"v","ρ":"p","α":"a","ε":"e","τ":"t","υ":"u","Α":"A",
    "Β":"B","Ε":"E","Η":"H","Ι":"I","Κ":"K","Μ":"M","Ν":"N","Ο":"O",
    "Ρ":"P","Τ":"T","Υ":"Y","Χ":"X","Ζ":"Z",
    # 풀너비 (full-width)
    "Ａ":"A","Ｂ":"B","Ｅ":"E","Ｈ":"H","Ｔ":"T",
}


def is_pure_ascii(s: str) -> bool:
    return all(ord(c) < 128 for c in s)


def normalize_lookalike(s: str) -> str:
    return "".join(LOOKALIKE_MAP.get(c, c) for c in s)


def classify_token(symbol: str, real_brands: set[str]) -> str:
    if is_pure_ascii(symbol):
        return "AUTHENTIC" if symbol in real_brands else "UNKNOWN_ASCII"
    # 라틴화
    norm = normalize_lookalike(symbol)
    # 트레일링 노이즈 제거 ("ЕТН..." -> "ETH")
    norm_stripped = norm.rstrip(" .\u2026\u00b7-_~`\'\"")
    if norm in real_brands or norm_stripped in real_brands:
        return "🚨 SPOOFED"
    return "OTHER_NON_ASCII"


# 실제 ETH/스테이킹 관련 브랜드
REAL_BRANDS = {"ETH", "stETH", "WETH", "wstETH", "rETH", "cbETH"}

erc20_classified = erc20_df.copy()
erc20_classified["verdict"] = erc20_classified["asset"].apply(
    lambda s: classify_token(s, REAL_BRANDS)
)

print("=== 토큰 심볼 검증 결과 ===")
print(erc20_classified["verdict"].value_counts())
print()


In [ ]:
print("=== 진짜 자금 흐름 (AUTHENTIC) ===")
real_flows = erc20_classified[erc20_classified["verdict"] == "AUTHENTIC"]
display(real_flows)


In [ ]:
print("=== 위장 토큰 (Address Poisoning, 무시할 노이즈) ===")
spoofed = erc20_classified[erc20_classified["verdict"] == "🚨 SPOOFED"]
display(spoofed.head(10))   # 보통 수십 건 → 첫 10건만
print(f"\n총 위장 토큰 흐름: {len(spoofed)} 건")


### 교훈

체인의 *데이터*는 진실이지만, *해석*은 신뢰할 수 없다. 누구나 임의 토큰을 발행해 임의 주소를 'from' 으로 표시할 수 있다.
**라벨, 심볼, 이벤트는 누구나 만들 수 있다 — 행위의 신뢰 근거는 컨트랙트의 *bytecode 자체*이지, 그 컨트랙트가 자칭하는 심볼이 아니다.**


## Step 4 — OFAC 제재 주소와 1-hop 대조 (Lab 2)

OFAC 의 디지털 자산 제재 리스트를 가져와, 방금 추적한 1-hop 수신자들과 대조한다.

**데이터 소스**: 0xB10C 의 [ofac-sanctioned-digital-currency-addresses](https://github.com/0xB10C/ofac-sanctioned-digital-currency-addresses) — OFAC SDN XML 에서 매일 0 UTC 자동 추출.


In [ ]:
OFAC_REPO_RAW = (
    "https://raw.githubusercontent.com/0xB10C/"
    "ofac-sanctioned-digital-currency-addresses/lists"
)
ASSETS = ["ETH", "XBT", "USDT", "USDC"]


@dataclass(frozen=True)
class SanctionEntry:
    address: str
    asset: str
    def __str__(self):
        return f"[{self.asset}] {self.address}"


def fetch_sanctioned(asset, timeout=30):
    url = f"{OFAC_REPO_RAW}/sanctioned_addresses_{asset}.txt"
    r = requests.get(url, timeout=timeout); r.raise_for_status()
    return [l.strip() for l in r.text.splitlines() if l.strip() and not l.startswith("#")]


def fetch_all_sanctions():
    out = []
    for a in ASSETS:
        try:
            addrs = fetch_sanctioned(a)
            print(f"  [{a}] {len(addrs)} 개")
            out.extend(SanctionEntry(addr, a) for addr in addrs)
        except requests.HTTPError as e:
            print(f"  [{a}] 다운로드 실패: {e}")
    return out


def screen_address(address, sanctions):
    a = address.lower()
    for e in sanctions:
        if e.address.lower() == a:
            return e
    return None


print("[*] OFAC 디지털 자산 제재 주소 다운로드 중...")
sanctions = fetch_all_sanctions()
print(f"\n[+] 총 {len(sanctions)} 개 제재 주소 로드 완료\n")

# Lab 1 의 1-hop 수신자에 대해 검사
candidates = set(eth_df["receiver"]) | set(erc20_df["receiver"])
print(f"1-hop 수신자: {len(candidates)} 개\n")

hits_1hop = [(a, screen_address(a, sanctions)) for a in candidates]
hits_1hop = [(a, h) for (a, h) in hits_1hop if h]

if hits_1hop:
    print(f"🚨 OFAC 매치 {len(hits_1hop)} 건:")
    for addr, e in hits_1hop:
        print(f"  {addr}  →  {e}")
else:
    print("✅ 1-hop 수신자 중 OFAC 매치 0건.")
    print()
    print("   📌 강의 메시지:")
    print("      Lazarus 는 OFAC 등재 *전* 의 신선한 EOA 로 1차 분산한다.")
    print("      단순 blacklist 만으로는 14.6억 달러가 *완벽하게 통과* 했을 것이다.")
    print("      → 다음 Step 들에서 행동 패턴·그래프 추적이 왜 필요한지 확인한다.")


## Step 5 — 행동 패턴 휴리스틱 (Lab 3)

OFAC 매치가 0건이었다. 그러면 이 자금 흐름이 *왜* 의심스러운지를 어떻게 데이터로 보일 것인가?
**행동 패턴**으로 보일 수 있다 — 라운드 수치, 시간 집중, 가스 동질성, peeling chain.

이번엔 합성 데이터가 아니라 **방금 추적한 실제 Lab 1 결과**에 적용한다.


In [ ]:
import math
from datetime import datetime, timezone, timedelta

# Lab 3 의 휴리스틱들
ROUND_VALUES_ETH = {0.01, 0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0, 100000.0}


def is_round_value(v: float, tol_rel: float = 1e-4) -> bool:
    """라운드 단위 (소수점 4자리 허용) 인가."""
    for r in ROUND_VALUES_ETH:
        if r > 0 and abs(v - r) / r < tol_rel:
            return True
    return False


# Lab 1 의 ETH 출금 데이터프레임을 그대로 평가
print("=== Lab 1 ETH 출금에 대한 휴리스틱 평가 ===\n")

# 1) 라운드 단위
round_count = sum(1 for amt in eth_df["total"] if is_round_value(amt))
print(f"라운드 단위 트랜잭션: {round_count} / {len(eth_df)}")
if round_count >= 5:
    print("  → 자동화된 분배의 강한 신호 (사람은 이렇게 깔끔한 숫자를 쓰지 않는다)")

# 2) 시간 집중도 — first_ts 의 분산
ts_seconds = pd.to_datetime(eth_df["first_ts"]).astype("int64") // 10**9
if len(ts_seconds) >= 2:
    span_sec = ts_seconds.max() - ts_seconds.min()
    span_min = span_sec / 60
    print(f"\n시간 집중도: {len(eth_df)} 건이 {span_min:.1f} 분 안에 발생")
    if span_min < 60 and len(eth_df) >= 10:
        print(f"  → 1시간 안에 10건 이상 = 사람 손이 아닌 봇/스크립트")

# 3) 금액 동질성 — 모두 같은 값이면 분명한 자동화
if len(eth_df) >= 3:
    amounts = eth_df["total"].values
    mean = amounts.mean()
    if mean > 0:
        cv = amounts.std() / mean   # 변동계수
        print(f"\n금액 변동계수 (CV): {cv:.4f}")
        if cv < 0.05:
            print(f"  → CV < 0.05 = 거의 동일한 금액으로 분배. *완벽한* 자동화 흔적.")
        elif cv < 0.2:
            print(f"  → 분산이 작음. 자동화 의심.")


In [ ]:
# 종합 위험도 점수 (실데이터 버전)
def assess_real(eth_df: pd.DataFrame) -> dict:
    flags = []
    score = 0.0

    if len(eth_df) == 0:
        return {"risk": 0.0, "flags": ["데이터 없음"]}

    # round
    rc = sum(1 for amt in eth_df["total"] if is_round_value(amt))
    if rc / max(1, len(eth_df)) >= 0.5:
        score += 0.30
        flags.append(f"라운드 단위 비율 {rc}/{len(eth_df)} ≥ 50%")

    # time concentration
    ts = pd.to_datetime(eth_df["first_ts"]).astype("int64") // 10**9
    if len(ts) >= 5:
        span_min = (ts.max() - ts.min()) / 60
        if span_min < 60 and len(ts) >= 10:
            score += 0.30
            flags.append(f"시간 집중: {len(ts)}건 / {span_min:.1f}분")

    # amount homogeneity
    if len(eth_df) >= 3:
        m = eth_df["total"].mean()
        if m > 0:
            cv = eth_df["total"].std() / m
            if cv < 0.05:
                score += 0.25
                flags.append(f"금액 동질성 (CV={cv:.4f})")

    # fan-out
    if len(eth_df) >= 20:
        score += 0.15
        flags.append(f"신규 EOA 분산: {len(eth_df)}개")

    return {"risk": min(1.0, score), "flags": flags}


result = assess_real(eth_df)
print(f"종합 위험도: {result['risk']:.0%}")
print("탐지된 신호:")
for f in result["flags"]:
    print(f"  - {f}")

print()
print("📌 강의 메시지:")
print(f"   OFAC 매치: 0%   /   행동 패턴 위험도: {result['risk']:.0%}")
print("   → blacklist 는 0점인데 행동 패턴은 정확히 탐지한다.")
print("   → 이것이 거래소·체이널리시스가 사용하는 *behavioral analytics* 의 핵심.")


## Step 6 — 2-hop 추적: 1차 mule 들은 다음 어디로?

OFAC blacklist 가 1-hop 에선 0건이었다. 더 멀리 가면 어떻게 되는가?
1차 mule 중 5개를 표본으로 골라 *그들의* 출금을 추적한다.

> 시간 절약: 5개만 추적. 전체 40개를 다 하면 무료 API 한도 (5 calls/sec) 에 걸린다.
> 강의 후 학생 숙제로 전체 확장 권장.


In [ ]:
SAMPLE_SIZE = 5
first_hop_addresses = list(set(eth_df["receiver"]))
sample = first_hop_addresses[:SAMPLE_SIZE]

print(f"1-hop mule 중 {SAMPLE_SIZE} 개 표본 추적:\n")

second_hop_eth_dfs = []
second_hop_receivers: set[str] = set()

for i, mule in enumerate(sample, 1):
    print(f"[{i}/{SAMPLE_SIZE}] {mule}")
    try:
        m_eth, m_erc20 = trace_outbound(mule, top_n=20)
        new_recv = set(m_eth["receiver"]) | set(m_erc20["receiver"])
        second_hop_receivers.update(new_recv)
        print(f"   → ETH 수신자 {len(m_eth)} + ERC-20 수신자 {len(m_erc20)} = {len(new_recv)} 개 분기")

        # 그래프용으로 ETH 흐름 기록 (mule -> receivers)
        if not m_eth.empty:
            tagged = m_eth.copy()
            tagged["src"] = mule
            second_hop_eth_dfs.append(tagged)
    except Exception as e:
        print(f"   ! 오류: {e}")
    time.sleep(0.3)

print(f"\n총 2-hop 수신자: {len(second_hop_receivers)} 개")


## Step 7 — 2-hop 결과를 OFAC 와 다시 대조

In [ ]:
hits_2hop = [(a, screen_address(a, sanctions)) for a in second_hop_receivers]
hits_2hop = [(a, h) for (a, h) in hits_2hop if h]

if hits_2hop:
    print(f"🚨 2-hop 에서 OFAC 매치 {len(hits_2hop)} 건 발견!")
    for addr, e in hits_2hop:
        print(f"  {addr}  →  {e}")
    print()
    print("📌 강의 메시지:")
    print("   여기서 등재된 컨트랙트(Tornado, Sinbad 등)와 만나기 시작한다.")
    print("   1-hop 에선 안 보였지만 2-hop 에서 그물에 걸린다 — 그래프 추적의 가치.")
else:
    print(f"2-hop ({len(second_hop_receivers)}개) 에서도 매치 0건.")
    print()
    print("📌 강의 메시지:")
    print("   Lazarus 는 2-hop 에서도 또 새 EOA 로 분산한다.")
    print("   보통 4~6 hop, 또는 cross-chain bridge 통과 후에야 등재 컨트랙트와 만난다.")
    print("   THORChain, eXch 같은 KYC 없는 브릿지가 그 단절점이다.")
    print("   → 학생 숙제: SAMPLE_SIZE 를 키우거나 hop 깊이를 늘려서 어디서 매치가 처음 나타나는지 찾아보라.")


## Step 8 — 자금 흐름 그래프 (Lab 6)

이제까지의 1-hop + 2-hop 데이터를 *방향 가중 그래프*로 시각화한다.
Lazarus 의 peeling chain 이 어떻게 보이는지 한 그림으로.


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

g = nx.DiGraph()

# 1-hop ETH 흐름 (seed → mule)
for _, row in eth_df.iterrows():
    g.add_edge(SEED_ADDRESS, row["receiver"], weight=float(row["total"]))

# 2-hop ETH 흐름 (mule → 그 다음)
for df in second_hop_eth_dfs:
    for _, row in df.iterrows():
        if g.has_edge(row["src"], row["receiver"]):
            g[row["src"]][row["receiver"]]["weight"] += float(row["total"])
        else:
            g.add_edge(row["src"], row["receiver"], weight=float(row["total"]))

print(f"노드 {g.number_of_nodes()}, 엣지 {g.number_of_edges()}")

# 노드 색상: seed 빨강, 1-hop mule 주황, 2-hop 회색
HOP1 = set(eth_df["receiver"])
def node_color(n):
    if n.lower() == SEED_ADDRESS.lower(): return "#d62728"  # red
    if n in HOP1: return "#ff9f4a"                          # orange
    return "#cfd8dc"                                        # gray

# 노드 라벨: seed 와 mule 만 짧게
def node_label(n):
    if n.lower() == SEED_ADDRESS.lower(): return "Bybit\nExploiter"
    return n[:6] + "…" + n[-4:]   # 0x47666f…86e2

# 중심성: 입출 차수
in_deg  = sorted(g.in_degree(),  key=lambda kv: -kv[1])[:5]
out_deg = sorted(g.out_degree(), key=lambda kv: -kv[1])[:5]
print("\nTop in-degree (가장 많이 받음):")
for n, d in in_deg:  print(f"  {node_label(n):<22}  ← {d}")
print("\nTop out-degree (가장 많이 보냄):")
for n, d in out_deg: print(f"  {node_label(n):<22}  → {d}")


In [ ]:
# 시각화 — 너무 많으면 표본만
MAX_NODES_TO_DRAW = 80
if g.number_of_nodes() <= MAX_NODES_TO_DRAW:
    sub = g
else:
    # seed + 1-hop + 2-hop 의 일부
    keep = {SEED_ADDRESS} | HOP1 | set(list(set(g.nodes()) - HOP1 - {SEED_ADDRESS})[:MAX_NODES_TO_DRAW - 1 - len(HOP1)])
    sub = g.subgraph(keep).copy()
    print(f"노드가 너무 많아 {sub.number_of_nodes()}개만 그립니다.")

pos = nx.spring_layout(sub, seed=42, k=0.6, iterations=50)
weights = [sub[u][v]["weight"] for u, v in sub.edges()]
max_w = max(weights) if weights else 1

plt.figure(figsize=(13, 9))
nx.draw_networkx_nodes(sub, pos,
                       node_size=[480 if n == SEED_ADDRESS else 220 for n in sub.nodes()],
                       node_color=[node_color(n) for n in sub.nodes()],
                       edgecolors="#333", linewidths=0.7)
nx.draw_networkx_edges(sub, pos,
                       edge_color="#888",
                       width=[0.5 + 3*(w/max_w) for w in weights],
                       arrows=True, arrowsize=8,
                       connectionstyle="arc3,rad=0.04")
labels = {n: node_label(n) for n in sub.nodes() if n == SEED_ADDRESS or n in HOP1}
nx.draw_networkx_labels(sub, pos, labels=labels, font_size=7)
plt.title(f"Bybit Exploiter peeling chain (1-hop + 2-hop sample of {SAMPLE_SIZE} mules)\n"
          f"red=seed  /  orange=1-hop mules  /  gray=2-hop",
          fontsize=10)
plt.axis("off")
plt.tight_layout()
plt.show()


## Step 9 — 강의 마무리

### 우리가 데이터로 본 것

1. **자금 추적**: 401,000 ETH 가 약 40개 EOA 로 정확히 10,000 ETH 씩 *5분* 안에 분산됐다.
2. **노이즈**: ERC-20 표의 절반 이상은 키릴 문자로 위장한 가짜 토큰이었다 (Address Poisoning).
3. **Blacklist 한계**: 1-hop, 2-hop 모두 OFAC 매치는 거의 0건. Lazarus 는 등재 *전* 신선한 EOA 를 사용한다.
4. **행동 패턴 탐지**: 라운드 수치 + 시간 집중 + 금액 동질성 = 위험도 매우 높음.
5. **그래프 추적**: peeling chain 의 fan-out 구조가 한 그림에 나타난다.

### 강의의 한 줄 요약

> **Blacklist 는 사후약방문이다. Lazarus 는 그 위에서 춤춘다.**
>
> 진짜 방어선은 (1) 위협 인텔리전스 피드, (2) 행동 패턴 탐지, (3) 그래프 추적, (4) 거래소 간 정보 공유.

### 학생 숙제 (난이도 순)

1. **확장**: `SAMPLE_SIZE` 를 5 → 40 으로 늘려 1차 mule 전체를 2-hop 추적하라. OFAC 매치가 처음 나타나는 곳은 어디인가?
2. **3-hop**: 같은 패턴으로 3-hop 까지 확장하라. 그래프가 폭발적으로 커지는데, 어떻게 *흥미로운* 노드만 골라낼 것인가? (힌트: 입출 차수, 컨트랙트 vs EOA 구분)
3. **체이널리시스 비교**: TRM Labs, Elliptic, Chainalysis 가 공개한 Bybit 사건 분석 보고서와 우리 그래프를 비교하라. 무엇이 다른가? 왜 다른가?
4. **실시간 시스템**: 위 코드를 cron 으로 매 분 실행해 *새 mule 이 등장하는 즉시* 알림을 보내려면 어떻게 설계할 것인가? (메시지 큐, 변경 감지, false positive 처리)

---

### 참고

- FBI PSA (2025-02-26): `$1.5B Bybit Hack - North Korea TraderTraitor`
- Safe{Wallet} 사후 분석 (2025-02-26)
- Chainalysis Crypto Crime Report (연간)
- 0xB10C OFAC SDN 추출본: https://github.com/0xB10C/ofac-sanctioned-digital-currency-addresses
- MITRE ATT&CK Group G0032 — Lazarus Group
